In [1]:
!nvidia-smi

Wed Oct  7 06:25:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.18.0" peft accelerate bitsandbytes datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 30.1 MB/s eta 0:00:00


In [3]:
import unsloth
print("Unsloth installed successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer


Unsloth installed successfully!


In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "prithvi1029/medquad-medical-qa",
    split="train"
)

print(dataset)

README.md:   0%|          | 0.00/842 [00:00<?, ?B/s]

medquad-cleaned-qa.csv: reconstructing file:   0%|          |  0.00B / 22.5MB            

medquad-cleaned-qa.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

Dataset({
    features: ['qtype', 'Question', 'Answer'],
    num_rows: 16407
})


In [5]:
print(dataset[0])

{'qtype': 'susceptibility', 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?', 'Answer': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'}


In [6]:
print(dataset.column_names)

['qtype', 'Question', 'Answer']


In [7]:
print("Total examples:", len(dataset))

Total examples: 16407


In [8]:
print(dataset.features)

{'qtype': Value('string'), 'Question': Value('string'), 'Answer': Value('string')}


In [9]:
# Shuffle the dataset
dataset = dataset.shuffle(seed=42)

# Keep 2,000 examples for our first experiment
small_dataset = dataset.select(range(2000))

# Split into training and testing sets
split_dataset = small_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Testing examples:", len(test_dataset))

Training examples: 1800
Testing examples: 200


In [11]:
def format_medical_example(example):
    return {
        "text": (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>\n\n"
            "You are a helpful medical information assistant."
            "<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            + example["Question"]
            + "<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
            + example["Answer"]
            + "<|eot_id|>"
        )
    }

print("Function created successfully!")

Function created successfully!


In [12]:
formatted_example = format_medical_example(train_dataset[0])

print(formatted_example["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful medical information assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What are the symptoms of Familial partial lipodystrophy associated with PPARG mutations ?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

What are the signs and symptoms of Familial partial lipodystrophy associated with PPARG mutations? The Human Phenotype Ontology provides the following list of signs and symptoms for Familial partial lipodystrophy associated with PPARG mutations. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Abnormality of lipid metabolism 90% Abnormality of the menstrual cycle 90% Diabetes mellitus 90% Hypertension 90% Insulin resistance 90% Acanthosis nigricans 50% 

In [13]:
from unsloth import FastLanguageModel

max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2026.10.1: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


In [14]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.10.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [15]:
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [18]:
def formatting_prompts_func(examples):
    conversations = []

    for question, answer in zip(
        examples["Question"],
        examples["Answer"]
    ):
        conversations.append([
            {
                "role": "system",
                "content": "You are a helpful medical information assistant."
            },
            {
                "role": "user",
                "content": question
            },
            {
                "role": "assistant",
                "content": answer
            }
        ])

    texts = []

    for conversation in conversations:
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return {"text": texts}

In [19]:
train_dataset = train_dataset.map(
    formatting_prompts_func,
    batched=True
)

test_dataset = test_dataset.map(
    formatting_prompts_func,
    batched=True
)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [20]:
print(train_dataset[0]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful medical information assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What are the symptoms of Familial partial lipodystrophy associated with PPARG mutations ?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

What are the signs and symptoms of Familial partial lipodystrophy associated with PPARG mutations? The Human Phenotype Ontology provides the following list of signs and symptoms for Familial partial lipodystrophy associated with PPARG mutations. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Abnormality of lipid metabolism 90% Abnormality of the menstrual cycle 90% Diabetes mellitus 90% Hy

In [23]:
from trl import SFTTrainer, SFTConfig

print("SFTTrainer loaded successfully!")

SFTTrainer loaded successfully!


In [24]:
training_args = SFTConfig(
    output_dir="medical-qlora-output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    fp16=True,
    optim="adamw_8bit",
    report_to="none",
    seed=42,
)

print("Training configuration created successfully!")

Training configuration created successfully!


In [25]:
import inspect

print("SFTConfig signature:")
print(inspect.signature(SFTConfig))

print("\nSFTTrainer signature:")
print(inspect.signature(SFTTrainer))

SFTConfig signature:
(output_dir=None, per_device_train_batch_size=4, num_train_epochs=3.0, max_steps=-1, learning_rate=5e-05, lr_scheduler_type='linear', lr_scheduler_kwargs=None, warmup_steps=0.1, optim='adamw_8bit', optim_args=None, weight_decay=0.001, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=2, average_tokens_across_devices=True, max_grad_norm=1.0, label_smoothing_factor=0.0, bf16=False, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=False, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=250, auto_find_batch_size=False, logging_strategy='steps', logging_steps=1, logging_first_step=False, log_on_each_node=True, logging_nan_inf_filter=False, include_num_input_tokens_seen=False, log_level='passive', 

In [26]:
training_args = SFTConfig(
    output_dir="medical-qlora-output",

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,

    # Sequence length
    max_length=1024,

    # Dataset
    dataset_text_field="text",

    # Evaluation
    eval_strategy="steps",
    eval_steps=100,
    per_device_eval_batch_size=1,

    # Logging & saving
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,

    # T4 ke liye FP16
    fp16=True,

    # Memory efficient optimizer
    optim="adamw_8bit",

    # Gradient checkpointing
    gradient_checkpointing=True,

    # Reproducibility
    seed=42,

    # No external logging
    report_to="none",
)

print("Training configuration ready!")

Training configuration ready!


In [27]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
)

print("SFTTrainer created successfully!")

/content/unsloth_compiled_cache/UnslothSFTTrainer.py:973: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your attention mechanism can handle flattened sequences.
  warnings.warn(


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1800 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
SFTTrainer created successfully!


In [28]:
!nvidia-smi

Wed Oct  7 06:54:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P0             26W /   70W |    5821MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [29]:
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved memory:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU: Tesla T4
Allocated memory: 5.51 GB
Reserved memory:  5.54 GB


In [30]:
train_result = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,800 | Num Epochs = 2 | Total steps = 900
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.940659,0.933890
200,0.856099,0.892896
300,0.857027,0.873361
400,0.893673,0.863150


Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-400/tokenizer_config.json.


Step,Training Loss,Validation Loss
100,0.940659,0.933890
200,0.856099,0.892896
300,0.857027,0.873361
400,0.893673,0.863150
500,0.696510,0.872430
600,0.792169,0.862224
700,0.662208,0.855932
800,0.781651,0.849666
900,0.664847,0.849969


Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-900/tokenizer_config.json.


In [31]:
final_model_path = "medical-qlora-final"

model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Model saved successfully to: {final_model_path}")

Unsloth: Restored added_tokens_decoder metadata in medical-qlora-final/tokenizer_config.json.


Model saved successfully to: medical-qlora-final


In [32]:
import os

print(os.listdir("medical-qlora-final"))

['chat_template.jinja', 'adapter_config.json', 'README.md', 'tokenizer.json', 'adapter_model.safetensors', 'tokenizer_config.json']


In [33]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

print("Model is ready for inference!")

Model is ready for inference!


In [34]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical information assistant."
    },
    {
        "role": "user",
        "content": "What are the common symptoms of asthma?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(
    outputs[0][inputs.shape[-1]:],
    skip_special_tokens=True
)

print("Model Response:\n")
print(response)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Model Response:

Common symptoms of asthma include:
                
- wheezing  - coughing  - chest tightness  - shortness of breath or trouble breathing
